In [1]:
import torch
import nibabel as nib
from nilearn.image import resample_to_img
from transformers import AutoModelForCausalLM, AutoTokenizer
from neurovlm.data import load_masker, load_dataset
from neurovlm.models import load_model
from neurovlm.retrieval_resources import NEURO_QWEN_REPO_ID

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32

masker = load_masker()
imgs = load_dataset("networks")

# NeuroQformer contains the _yy.pt Q-Former tensors together with the
# matching image projection head and 907-row canonical semantic banks.
qformer = load_model("neuro_qformer").to(device).eval()
# The historical notebook cast only the trained Q-Former to bfloat16.
# Keep the packaged projection head and canonical banks in float32.
qformer.qformer.to(dtype=model_dtype)
qformer_dtype = next(qformer.qformer.parameters()).dtype
autoencoder = load_model("autoencoder").to(device).eval()
encoder = autoencoder.encoder

tokenizer = AutoTokenizer.from_pretrained(NEURO_QWEN_REPO_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    NEURO_QWEN_REPO_ID,
    torch_dtype=model_dtype,
).to(device).eval()

with torch.no_grad():
    lm_token_norm = (
        model.get_input_embeddings().weight.detach().float().norm(dim=1).mean()
    ).item()

In [5]:
refs = [
    ["YeoLab", "VisualB"],
    ["Glasser", "Language"],
    ['Shirer', 'Auditory'],
    ["WashU", "LateralSM"],
    ["Laird", "Emo/Interoception1"],
    ["Shen", "default mode"],
    ["Du", "FPN-A"],
    ["Du", "SAL/PMN"],
]

image_tensors = []
for item in refs:
    if len(item) == 1:
        continue
    atlas, map_name = item

    im = nib.Nifti1Image(imgs[atlas][map_name]["array"] / imgs[atlas][map_name]["array"].max(), imgs[atlas][map_name]["affine"])

    t = torch.from_numpy(masker.transform(resample_to_img(im, masker.mask_img, interpolation="nearest")))

    image_tensors.append(t)

In [6]:
do_sample = False
temperature = None
top_p = None
num_beams = 5
max_new_tokens = 256
generations = []

for (atlas_name, map_name), t in zip(refs, image_tensors):
    print(f"{atlas_name}_{map_name}")

    for seed, basis in enumerate(["network", "region", "function"]):
        with torch.no_grad():
            raw_latent = encoder(t.to(device)).detach().float()
            raw_latent = raw_latent.reshape(1, -1)

            # Match the producer notebook's precision boundary: image and
            # canonical projections run in float32, before LM autocast.
            semantic_latent = qformer.project_semantic(
                raw_latent,
                basis=basis,
                projection_temp=0.05,
                use_canonical_projection=True,
            ).detach()
            raw_input = raw_latent.to(device=device, dtype=qformer_dtype)
            semantic_input = semantic_latent.to(
                device=device, dtype=qformer_dtype
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                vis = qformer.qformer(raw_input, semantic_input)

                target_norm = torch.tensor(
                    lm_token_norm, device=vis.device, dtype=vis.dtype
                )
                vis_norm = vis.float().norm(dim=-1, keepdim=True).clamp_min(1e-6)
                vis = vis * (target_norm / vis_norm.to(vis.dtype))
                vis = vis.to(model_dtype)

                prefix_text = f"[{basis.upper()}]"
                prefix_ids = tokenizer(
                    prefix_text,
                    add_special_tokens=False,
                    return_tensors="pt",
                )["input_ids"].to(device)
                prefix_embeds = model.get_input_embeddings()(prefix_ids).to(model_dtype)
                inputs_embeds = torch.cat([vis, prefix_embeds], dim=1)
                attention_mask = torch.ones(
                    inputs_embeds.shape[:2], dtype=torch.long, device=device
                )

                generation_kwargs = {
                    "inputs_embeds": inputs_embeds,
                    "attention_mask": attention_mask,
                    "max_new_tokens": max_new_tokens,
                    "num_beams": num_beams,
                    "do_sample": do_sample,
                    "repetition_penalty": 1.18,
                    "no_repeat_ngram_size": 4,
                    "eos_token_id": tokenizer.eos_token_id,
                    "pad_token_id": tokenizer.eos_token_id,
                    "top_k": None,
                    "temperature": temperature,
                    "top_p": top_p,
                }

                cuda_devices = (
                    [torch.cuda.current_device()] if device.type == "cuda" else []
                )
                with torch.random.fork_rng(devices=cuda_devices):
                    torch.manual_seed(seed)
                    if device.type == "cuda":
                        torch.cuda.manual_seed_all(seed)
                    output_ids = model.generate(**generation_kwargs)

            generated = tokenizer.decode(
                output_ids[0], skip_special_tokens=True
            ).strip()
            prediction = prefix_text + generated

        generations.append((atlas_name, map_name, prediction))
        print(prediction)
        print()

    print()

YeoLab_VisualB
[NETWORK]Visual Network
This network is broadly associated with visual processing. It includes the occipital, temporal, and parietal cortices. The network is generally thought to support the integration of visual information across the brain. It is often described as a central hub for visual perception.

[REGION]Fusiform Gyrus
This region is broadly associated with the perception of visual form and category. It is often linked to the early stages of visual processing. Its function is generally described as supporting the recognition of meaningful visual patterns.

[FUNCTION]Visual Perception
Visual perception is the process of interpreting visual information. It involves the occipital lobe, temporal lobe, parietal lobe, and visual cortex. This function is broadly linked to spatial awareness, object recognition, and the integration of visual and contextual information.


Glasser_Language
[NETWORK]Language Network
The language network includes posterior superior temporal c